# 🏅 Olympics Data Lake — Paris 2024

Este notebook realiza todo o pipeline do Data Lake:
1. **RAW → BRONZE**: Leitura dos CSVs e conversão para Parquet
2. **BRONZE → BRONZE**: JOINs entre datasets
3. **BRONZE → GOLD**: Análises e visualizações do quadro de medalhas

## ⚙️ 0. Instalação de dependências e imports

In [1]:
# Instale se necessário:
# !pip install pandas pyarrow matplotlib seaborn

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json, os
from datetime import datetime

# Caminhos base
BASE     = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
RAW_DIR    = os.path.join(BASE, 'raw')
BRONZE_DIR = os.path.join(BASE, 'bronze')
GOLD_DIR   = os.path.join(BASE, 'gold', 'analise_medalhas')

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(GOLD_DIR,   exist_ok=True)

print("Diretórios configurados:")
print(f"  RAW    → {RAW_DIR}")
print(f"  BRONZE → {BRONZE_DIR}")
print(f"  GOLD   → {GOLD_DIR}")

Diretórios configurados:
  RAW    → c:\Users\Thiago-PC\Desktop\faculdade\cienciasDeDados\exploracao-dados-thiago-pincegher-simoes\olympics-datalake\raw
  BRONZE → c:\Users\Thiago-PC\Desktop\faculdade\cienciasDeDados\exploracao-dados-thiago-pincegher-simoes\olympics-datalake\bronze
  GOLD   → c:\Users\Thiago-PC\Desktop\faculdade\cienciasDeDados\exploracao-dados-thiago-pincegher-simoes\olympics-datalake\gold\analise_medalhas


## 📥 1. RAW → BRONZE: Leitura e conversão para Parquet

In [2]:
DATASETS = [
    'athletes', 'coaches', 'events', 'medallists', 'medals',
    'medals_total', 'nocs', 'schedules', 'schedules_preliminary',
    'teams', 'technical_officials', 'torch_route', 'venues'
]

dfs = {}

for name in DATASETS:
    csv_path     = os.path.join(RAW_DIR, f'{name}.csv')
    parquet_path = os.path.join(BRONZE_DIR, f'{name}.parquet')
    
    if not os.path.exists(csv_path):
        print(f'⚠️  Arquivo não encontrado: {csv_path}')
        continue
    
    df = pd.read_csv(csv_path, low_memory=False)
    dfs[name] = df
    
    # Salva como Parquet
    df.to_parquet(parquet_path, index=False)
    print(f'✅ {name:30s} → {df.shape[0]:>5} linhas, {df.shape[1]:>2} colunas → .parquet')

print(f'\n✔ {len(dfs)} datasets convertidos para Parquet em bronze/')

✅ athletes                       → 11113 linhas, 36 colunas → .parquet
✅ coaches                        →   974 linhas, 12 colunas → .parquet
✅ events                         →   329 linhas,  5 colunas → .parquet
✅ medallists                     →  2315 linhas, 21 colunas → .parquet
✅ medals                         →  1044 linhas, 13 colunas → .parquet
✅ medals_total                   →    92 linhas,  7 colunas → .parquet
✅ nocs                           →   224 linhas,  5 colunas → .parquet
✅ schedules                      →  3895 linhas, 16 colunas → .parquet
✅ schedules_preliminary          →  2298 linhas, 18 colunas → .parquet
✅ teams                          →  1698 linhas, 16 colunas → .parquet
✅ technical_officials            →  1021 linhas, 11 colunas → .parquet
✅ torch_route                    →    73 linhas,  7 colunas → .parquet
✅ venues                         →    35 linhas,  6 colunas → .parquet

✔ 13 datasets convertidos para Parquet em bronze/


## 🔗 2. BRONZE: JOINs entre datasets

### 2.1 Medalistas Enriquecidos (medallists + athletes + nocs)

In [3]:
if 'medallists' in dfs and 'athletes' in dfs and 'nocs' in dfs:
    
    df_med = dfs['medallists'].copy()
    df_ath = dfs['athletes'][['name','height','weight','birth_date']].drop_duplicates(subset='name')
    df_noc = dfs['nocs'][['code','country_long']].rename(columns={'code':'country_code'})
    
    # JOIN
    df_enriched = (
        df_med
        .merge(df_ath, on='name', how='left')
        .merge(df_noc, on='country_code', how='left')
    )
    
    dfs['medallists_enriched'] = df_enriched
    df_enriched.to_parquet(os.path.join(BRONZE_DIR, 'medallists_enriched.parquet'), index=False)
    
    print(f"✅ medallists_enriched: {df_enriched.shape}")
    display(df_enriched.head(3))
else:
    print("⚠️  Datasets necessários não carregados.")

✅ medallists_enriched: (2315, 25)


,medal_date,medal_type,medal_code,name,gender,country_code,country,country_long_x,nationality_code,nationality,...,event_type,url_event,birth_date_x,code_athlete,code_team,is_medallist,height,weight,birth_date_y,country_long_y
0,2024-07-27,Gold Medal,1.0,EVENEPOEL Remco,Male,BEL,Belgium,Belgium,BEL,Belgium,...,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,2000-01-25,1903136,NaN,True,0.0,0.0,2000-01-25,Belgium
1,2024-07-27,Silver Medal,2.0,GANNA Filippo,Male,ITA,Italy,Italy,ITA,Italy,...,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,1996-07-25,1923520,NaN,True,0.0,0.0,1996-07-25,Italy
2,2024-07-27,Bronze Medal,3.0,van AERT Wout,Male,BEL,Belgium,Belgium,BEL,Belgium,...,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,1994-09-15,1903147,NaN,True,0.0,0.0,1994-09-15,Belgium


In [ ]:
if 'medals' in dfs and 'nocs' in dfs:
    
    df_noc = dfs['nocs'][['code','country_long']].rename(columns={'code':'country_code'})
    df_medals_country = dfs['medals_total'].merge(df_noc, on='country_code', how='left')
    
    dfs['medals_by_country'] = df_medals_country
    df_medals_country.to_parquet(os.path.join(BRONZE_DIR, 'medals_by_country.parquet'), index=False)
    
    print("✅ medals_by_country:")
    display(df_medals_country.head(5))

✅ medals_by_country:


,medal_type,medal_code,medal_date,name,gender,discipline,event,event_type,url_event,code,country_code,country,country_long_x,country_long_y
0,Gold Medal,1.0,2024-07-27,Remco EVENEPOEL,M,Cycling Road,Men's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,1903136,BEL,Belgium,Belgium,Belgium
1,Silver Medal,2.0,2024-07-27,Filippo GANNA,M,Cycling Road,Men's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,1923520,ITA,Italy,Italy,Italy
2,Bronze Medal,3.0,2024-07-27,Wout van AERT,M,Cycling Road,Men's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,1903147,BEL,Belgium,Belgium,Belgium
3,Gold Medal,1.0,2024-07-27,Grace BROWN,W,Cycling Road,Women's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/women-s-in...,1940173,AUS,Australia,Australia,Australia
4,Silver Medal,2.0,2024-07-27,Anna HENDERSON,W,Cycling Road,Women's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/women-s-in...,1912525,GBR,Great Britain,Great Britain,Great Britain


In [7]:
if 'schedules' in dfs and 'venues' in dfs:
    
    df_ven = dfs['venues'][['venue','sports']].rename(
        columns={'sports':'sports_in_venue'})
    df_sched_enr = dfs['schedules'].merge(df_ven, on='venue', how='left')
    
    dfs['schedules_enriched'] = df_sched_enr
    df_sched_enr.to_parquet(os.path.join(BRONZE_DIR, 'schedules_enriched.parquet'), index=False)
    
    print(f"✅ schedules_enriched: {df_sched_enr.shape}")
    display(df_sched_enr.head(3))

✅ schedules_enriched: (3895, 17)


,start_date,end_date,day,status,discipline,discipline_code,event,event_medal,phase,gender,event_type,venue,venue_code,location_description,location_code,url,sports_in_venue
0,2024-07-24T15:00:00+02:00,2024-07-24T16:45:00+02:00,2024-07-24,FINISHED,Football,FBL,Men,0,Men's Group B,M,HTEAM,Geoffroy-Guichard Stadium,STE,"Geoffroy-Guichard Stadium, Saint-Etienne",STE,/en/paris-2024/results/football/men/gpb-000100--,['Football']
1,2024-07-24T15:00:00+02:00,2024-07-24T16:45:00+02:00,2024-07-24,FINISHED,Football,FBL,Men,0,Men's Group C,M,HTEAM,Parc des Princes,PDP,"Parc des Princes, Paris",PDP,/en/paris-2024/results/football/men/gpc-000100--,['Football']
2,2024-07-24T15:30:00+02:00,2024-07-24T15:46:00+02:00,2024-07-24,FINISHED,Rugby Sevens,RU7,Men,0,Men's Pool B,M,HTEAM,Stade de France,STA,Stade de France,STA,/en/paris-2024/results/rugby-sevens/men/gpb-00...,"['Athletics', 'Rugby Sevens']"


## 📊 3. GOLD: Análise do Quadro de Medalhas

In [ ]:
if 'medals_total' in dfs and 'nocs' in dfs:
    df = dfs['medals_by_country'].copy()
    
    # Garante nomes corretos das colunas de medalha
    gold_col   = [c for c in df.columns if 'Gold'   in c][0]
    silver_col = [c for c in df.columns if 'Silver' in c][0]
    bronze_col = [c for c in df.columns if 'Bronze' in c][0]
    
    df = df.rename(columns={gold_col:'ouro', silver_col:'prata', bronze_col:'bronze'})
    df['total'] = df['ouro'] + df['prata'] + df['bronze']
    df = df.sort_values('ouro', ascending=False).reset_index(drop=True)
    df['rank'] = df.index + 1
    
    # Salva summary CSV
    summary_path = os.path.join(GOLD_DIR, 'medals_summary.csv')
    df.to_csv(summary_path, index=False)
    
    print("📋 Top 10 do Quadro de Medalhas — Paris 2024:")
    display(df[['rank','country','ouro','prata','bronze','total']].head(10))

IndexError: list index out of range

### 3.1 Visualização: Top 15 países por medalhas de ouro

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

top15 = df.head(15)
x = range(len(top15))

ax.bar([i-0.27 for i in x], top15['ouro'],   width=0.26, color='#FFD700', label='Ouro',   edgecolor='black', linewidth=0.5)
ax.bar([i      for i in x], top15['prata'],  width=0.26, color='#C0C0C0', label='Prata',  edgecolor='black', linewidth=0.5)
ax.bar([i+0.27 for i in x], top15['bronze'], width=0.26, color='#CD7F32', label='Bronze', edgecolor='black', linewidth=0.5)

ax.set_xticks(list(x))
ax.set_xticklabels(top15['country'], rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Número de Medalhas', fontsize=12)
ax.set_title('🏅 Top 15 Países — Quadro de Medalhas Paris 2024', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(axis='y', alpha=0.3)
sns.despine()

plt.tight_layout()
plot_path = os.path.join(GOLD_DIR, 'medals_plot.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Gráfico salvo em: {plot_path}")

### 3.2 Visualização: Distribuição de medalhas por gênero

In [ ]:
if 'medallists_enriched' in dfs:
    df_gen = dfs['medallists_enriched'].copy()
    
    gender_counts = df_gen.groupby(['medal_type', 'gender']).size().unstack(fill_value=0)
    gender_counts = gender_counts.reindex(['Gold Medal', 'Silver Medal', 'Bronze Medal'])
    
    colors = {'Male': '#4A90D9', 'Female': '#E05C8A', 'Mixed': '#7AC74F'}
    gender_counts.rename(index={
        'Gold Medal': 'Ouro', 'Silver Medal': 'Prata', 'Bronze Medal': 'Bronze'
    }).plot(kind='bar', figsize=(9, 5), color=[colors.get(c, '#999') for c in gender_counts.columns],
            edgecolor='black', linewidth=0.5)
    
    plt.title('Distribuição de Medalhas por Gênero — Paris 2024', fontsize=13, fontweight='bold')
    plt.xlabel('Tipo de Medalha')
    plt.ylabel('Quantidade')
    plt.xticks(rotation=0)
    plt.legend(title='Gênero')
    plt.grid(axis='y', alpha=0.3)
    sns.despine()
    plt.tight_layout()
    
    gender_plot_path = os.path.join(GOLD_DIR, 'gender_medals_plot.png')
    plt.savefig(gender_plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Gráfico de gênero salvo.")

### 3.3 Visualização: Top 10 modalidades com mais medalhas

In [ ]:
if 'medallists_enriched' in dfs:
    top_sports = (dfs['medallists_enriched']
                  .groupby('discipline')
                  .size()
                  .sort_values(ascending=False)
                  .head(10))
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(top_sports.index[::-1], top_sports.values[::-1],
                   color=sns.color_palette('viridis', 10)[::-1], edgecolor='black', linewidth=0.5)
    
    for bar, val in zip(bars, top_sports.values[::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=10)
    
    ax.set_title('Top 10 Modalidades com Mais Medalhas — Paris 2024', fontsize=13, fontweight='bold')
    ax.set_xlabel('Total de Medalhas')
    sns.despine()
    plt.tight_layout()
    
    sports_plot_path = os.path.join(GOLD_DIR, 'top_sports_medals_plot.png')
    plt.savefig(sports_plot_path, dpi=150, bbox_inches='tight')
    plt.show()

## 📝 4. Metadados da camada Gold

In [ ]:
gold_metadata = {
    "nome_dataset": "analise_medalhas",
    "descricao": "Análise completa do quadro de medalhas dos Jogos Olímpicos de Paris 2024, incluindo distribuição por país, gênero e modalidade.",
    "notebook": "notebook.ipynb",
    "data_geracao": datetime.now().strftime('%Y-%m-%d'),
    "fontes_bronze": ["medals_by_country", "medallists_enriched"],
    "arquivos_gerados": [
        "medals_summary.csv",
        "medals_plot.png",
        "gender_medals_plot.png",
        "top_sports_medals_plot.png"
    ],
    "metricas": {
        "total_paises_com_medalha": int(df['total'].gt(0).sum()) if 'df' in dir() else None,
        "total_medalhas_ouro": int(df['ouro'].sum()) if 'df' in dir() else None,
        "total_medalhas_geral": int(df['total'].sum()) if 'df' in dir() else None,
    },
    "observacoes": "Análise realizada com dados finais após encerramento dos Jogos Olímpicos de Paris 2024."
}

meta_path = os.path.join(GOLD_DIR, 'metadata.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(gold_metadata, f, ensure_ascii=False, indent=2)

print("✅ Metadados da camada Gold salvos!")
print(json.dumps(gold_metadata, indent=2, ensure_ascii=False))

## ✅ Pipeline concluído!

| Camada | Conteúdo |
|--------|----------|
| `raw/` | CSVs originais + JSONs de metadados |
| `bronze/` | Parquets convertidos + JOINs + JSONs de metadados |
| `gold/analise_medalhas/` | Resumo CSV + 3 gráficos PNG + metadados |